# LSTM Practical Examples — NETSOL Student Notebook

This notebook contains **three independent examples**:
1. IMDB movie-review sentiment analysis (binary classification)
2. Next-word prediction (language modeling)
3. Time-series forecasting (numerical regression)

**Requirements:** Python, TensorFlow/Keras, NumPy, and Matplotlib.

Install if needed:
```python
%pip install tensorflow numpy matplotlib
```

Run each example from top to bottom. The first run may download datasets. If the IMDB download is unavailable, the sentiment example falls back to a small synthetic dataset so the pipeline can still be demonstrated. The next-word example uses a small built-in teaching corpus, and the forecasting example generates synthetic data.

These examples are designed for teaching. Small datasets and short training runs are not intended to produce production-level accuracy.

---
# Example 1 — IMDB Sentiment Analysis

**Goal:** Predict whether a movie review is positive (1) or negative (0).

Pipeline: load reviews → pad sequences → Embedding → LSTM → sigmoid output.

The IMDB dataset is available through Keras. The code below attempts to download it; if that fails, it uses a small synthetic teaching dataset.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

VOCAB_SIZE = 10000
MAX_LEN = 200
BATCH_SIZE = 64
EPOCHS = 3

try:
    # Keras IMDB dataset: reviews are already encoded as integer word IDs.
    (x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
        num_words=VOCAB_SIZE
    )
    print("Loaded the IMDB dataset from Keras.")
    using_synthetic_imdb = False

except Exception as error:
    print("Could not download IMDB dataset:", error)
    print("Using a small synthetic dataset for demonstration.")

    positive_reviews = [
        "this movie was wonderful and excellent",
        "great acting and a fantastic story",
        "I loved this film it was enjoyable",
        "the plot was exciting and the actors were brilliant",
        "a beautiful movie with a great ending",
        "excellent film I would watch it again",
        "the story was touching and very good",
        "wonderful performances and amazing direction",
    ] * 20

    negative_reviews = [
        "this movie was terrible and boring",
        "the acting was poor and the story was awful",
        "I hated this film it was disappointing",
        "the plot was dull and the actors were bad",
        "a horrible movie with a terrible ending",
        "poor film I would not watch it again",
        "the story was confusing and very bad",
        "boring performances and awful direction",
    ] * 20

    texts = positive_reviews + negative_reviews
    labels = np.array([1] * len(positive_reviews) + [0] * len(negative_reviews))

    # A tiny tokenizer is sufficient for the fallback demonstration.
    tokenizer = keras.preprocessing.text.Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    padded = keras.utils.pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

    indices = np.random.permutation(len(padded))
    split = int(0.8 * len(indices))
    train_idx, test_idx = indices[:split], indices[split:]
    x_train, y_train = padded[train_idx], labels[train_idx]
    x_test, y_test = padded[test_idx], labels[test_idx]
    using_synthetic_imdb = True

# Pad every review to a fixed length so they can be placed in batches.
x_train = keras.utils.pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = keras.utils.pad_sequences(x_test, maxlen=MAX_LEN, padding="post", truncating="post")

print("Training reviews:", len(x_train))
print("Testing reviews:", len(x_test))
print("One padded review shape:", x_train[0].shape)

### Build and train the LSTM model

- `Embedding`: maps each word ID to a trainable dense vector.
- `LSTM(64)`: reads the sequence and returns the final output representation.
- `Dense(1, sigmoid)`: returns a probability between 0 and 1.

In [ ]:
sentiment_model = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=32, mask_zero=True),
    layers.LSTM(64),
    layers.Dense(1, activation="sigmoid")
])

sentiment_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

sentiment_model.summary()

history = sentiment_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

test_loss, test_accuracy = sentiment_model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

### Predict sentiment for a new review

For the real IMDB dataset, use the dataset's own word-index mapping. In the synthetic fallback, use the tokenizer created above.

In [ ]:
if not using_synthetic_imdb:
    word_index = keras.datasets.imdb.get_word_index()
    # IMDB convention: reserve 0 for padding, 1 for start, 2 for unknown.
    word_index = {word: index + 3 for word, index in word_index.items()}
    word_index["<PAD>"] = 0
    word_index["<START>"] = 1
    word_index["<UNK>"] = 2

    def encode_review(text):
        words = text.lower().split()
        ids = [1]
        for word in words:
            idx = word_index.get(word, 2)
            ids.append(idx if idx < VOCAB_SIZE else 2)
        return keras.utils.pad_sequences([ids], maxlen=MAX_LEN, padding="post", truncating="post")[0]

else:
    def encode_review(text):
        ids = tokenizer.texts_to_sequences([text])
        return keras.utils.pad_sequences(ids, maxlen=MAX_LEN, padding="post", truncating="post")[0]

def predict_sentiment(review):
    sequence = encode_review(review)
    probability = float(sentiment_model.predict(np.array([sequence]), verbose=0)[0][0])
    label = "Positive" if probability >= 0.5 else "Negative"
    print("Review:", review)
    print(f"Positive probability: {probability:.3f}")
    print("Predicted sentiment:", label)

predict_sentiment("The movie was wonderful and the acting was excellent.")
predict_sentiment("The movie was boring and the story was terrible.")

---
# Example 2 — Next-Word Prediction

**Goal:** Given a few words, predict the next word.

This example uses a small built-in teaching corpus, so no external dataset is required. It creates training samples such as:
- `I love` → `learning`
- `I love learning` → `Python`

The model predicts a probability distribution over the vocabulary; the code selects the most likely next word. With a tiny corpus, predictions are only for demonstration.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

corpus = [
    "i love learning python",
    "i love learning deep learning",
    "students learn machine learning",
    "students learn python programming",
    "i enjoy teaching students",
    "we are learning neural networks",
    "lstm models learn from sequences",
    "python is useful for data science",
    "students are building deep learning models",
    "we love building useful projects",
    "machine learning is interesting",
    "deep learning uses neural networks",
    "i am teaching lstm to students",
    "students practice python every day",
    "we are studying recurrent neural networks",
    "data science uses python programming",
]

tokenizer = keras.preprocessing.text.Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(corpus)
word_to_id = tokenizer.word_index
id_to_word = {idx: word for word, idx in word_to_id.items()}
vocab_size = len(word_to_id) + 1

# Turn each sentence into progressive input -> next-word examples.
sequences = []
for sentence in corpus:
    ids = tokenizer.texts_to_sequences([sentence])[0]
    for end in range(2, len(ids) + 1):
        sequences.append(ids[:end])

max_len = max(len(seq) for seq in sequences)
padded_sequences = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre")
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

print("Vocabulary size:", vocab_size)
print("Training examples:", len(X))
print("Input shape:", X.shape)
print("First input token IDs:", X[0])
print("Target token ID:", y[0], "=>", id_to_word.get(int(y[0]), "?"))

### Build and train the next-word model

The LSTM reads the prefix sequence. The softmax output gives a probability for every word in the vocabulary.

In [ ]:
next_word_model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=32, mask_zero=True),
    layers.LSTM(64),
    layers.Dense(vocab_size, activation="softmax")
])

next_word_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

next_word_model.summary()

history_next = next_word_model.fit(
    X, y,
    epochs=150,
    batch_size=8,
    verbose=0
)
print("Training complete.")

In [ ]:
def predict_next_word(text):
    ids = tokenizer.texts_to_sequences([text])[0]
    ids = ids[-(max_len - 1):]
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre")
    probabilities = next_word_model.predict(padded, verbose=0)[0]

    # Avoid selecting the padding/OOV token where possible.
    probabilities[0] = 0
    next_id = int(np.argmax(probabilities))
    next_word = id_to_word.get(next_id, "<unknown>")
    print(f"Input: {text!r}")
    print(f"Predicted next word: {next_word!r}")
    print(f"Model probability for this word: {probabilities[next_id]:.3f}")

predict_next_word("i love")
predict_next_word("students learn")
predict_next_word("deep learning")

---
# Example 3 — Time-Series Forecasting

**Goal:** Use previous numerical observations to predict the next value.

We will generate a synthetic daily series with a trend, a repeating seasonal pattern, and a little noise. This avoids needing an external file or API.

For each sample, the model sees the previous 30 values and predicts the next value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

# Synthetic daily data: trend + weekly seasonality + noise.
days = np.arange(1000)
series = (
    0.02 * days
    + 5 * np.sin(2 * np.pi * days / 7)
    + 0.8 * np.sin(2 * np.pi * days / 30)
    + np.random.normal(0, 0.7, size=len(days))
).astype("float32")

plt.figure(figsize=(12, 4))
plt.plot(days[:200], series[:200])
plt.title("Synthetic time series (first 200 days)")
plt.xlabel("Day")
plt.ylabel("Observed value")
plt.show()

WINDOW_SIZE = 30
split_point = int(len(series) * 0.8)

# Split chronologically before making samples to avoid leaking future values.
train_series = series[:split_point]
test_series = series[split_point - WINDOW_SIZE:]

def make_windows(values, window_size):
    X, y = [], []
    for i in range(window_size, len(values)):
        X.append(values[i-window_size:i])
        y.append(values[i])
    X = np.array(X, dtype="float32")[..., np.newaxis]
    y = np.array(y, dtype="float32")
    return X, y

X_train, y_train = make_windows(train_series, WINDOW_SIZE)
X_test, y_test = make_windows(test_series, WINDOW_SIZE)

print("X_train shape:", X_train.shape)  # (samples, 30 time steps, 1 feature)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

### Build and train the forecasting LSTM

This is regression, not classification. Therefore, the final Dense layer has one unit with no sigmoid activation, and we use mean squared error.

In [ ]:
forecast_model = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, 1)),
    layers.LSTM(32),
    layers.Dense(1)
])

forecast_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

forecast_model.summary()

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_forecast = forecast_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_mae = forecast_model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

In [ ]:
# Compare predictions with actual held-out values.
predictions = forecast_model.predict(X_test, verbose=0).flatten()

plt.figure(figsize=(12, 4))
plt.plot(y_test[:150], label="Actual")
plt.plot(predictions[:150], label="Predicted")
plt.title("LSTM forecasting: actual vs predicted")
plt.xlabel("Test time step")
plt.ylabel("Value")
plt.legend()
plt.show()

# Predict the next value after the end of the full observed series.
last_window = series[-WINDOW_SIZE:][np.newaxis, :, np.newaxis]
next_value = float(forecast_model.predict(last_window, verbose=0)[0][0])
print(f"Predicted next value: {next_value:.3f}")

---
## Student checklist

- [ ] Explain why sequences need padding in the sentiment example.
- [ ] Explain why the sentiment model uses sigmoid and binary cross-entropy.
- [ ] Explain why next-word prediction uses softmax and sparse categorical cross-entropy.
- [ ] Explain why forecasting uses a linear Dense output and MSE.
- [ ] Change the LSTM unit count (e.g., 32, 64, 128) and compare results.
- [ ] For forecasting, change the window size from 30 to 14 or 60.

**Important:** The next-word example intentionally uses a tiny corpus to keep the code understandable. A useful language model requires a much larger and more varied dataset.